# LCA Lasers4MaaS: pre- and post-joining processes

Unit processes of the chemical and mechanical treatments before and after joining, per material (steel, aluminium).
The functional unit of each process is 1 execution of the process.

The calculation code is in the `l4m_lca` package. This notebook holds the parameters and shows the results.

In [1]:
import bw2data as bd
from IPython.display import display

from l4m_lca import plots
from l4m_lca.constants import (CHEM_PREPOST_DATABASE, EU27_GB, EUROSTAT_ELEC_PRICE_2025S1, MECH_PREPOST_DATABASE,
                               PREPOST_STEM, REGIONAL_BASELINE, REGIONAL_ELEC_PRICE, SCENARIOS)
from l4m_lca.impacts import BackgroundImpacts
from l4m_lca.costs import COST_UNITS, LABOUR_FIELD, regional_labour_costs
from l4m_lca.lcia import export, scenario_table
from l4m_lca.prepost import (TIME_PARAMETERS, chemical_exchanges, mechanical_exchanges, per_material, prepost_labour,
                             process_code, process_name)
from l4m_lca.project import (DEFAULT_INDICATORS, category, ef31_methods, find_activity, method_units, open_project,
                             select_background)
from l4m_lca.regional import electricity_markets, regional_table

## Parameters

In [2]:
# Background database: "ecoinvent" or "bafu". The environment variable L4M_BACKGROUND overrides it.
BG = select_background("ecoinvent")
BACKGROUND = BG.name

INDICATORS = DEFAULT_INDICATORS

# Background activities of ecoinvent 3.12: (name, location, reference product or None)
ECOINVENT_FLOWS = {
    "electricity":        ("market group for electricity, medium voltage", "RER", None),          # kWh
    "water":              ("market group for tap water", "RER", None),                            # kg
    "sodium_hydroxide":   ("market for sodium hydroxide, without water, in 50% solution state", "RER", None),
    "hydrochloric_acid":  ("market for hydrochloric acid, without water, in 30% solution state", "RER", None),
    "nitric_acid":        ("market for nitric acid, without water, in 50% solution state", "RER w/o RU", None),
    # Mechanical flows are PROXIES, all in kg. ecoinvent 3.12 has no "grinding wheel" or
    # "metal working fluid" dataset, and "abrasive blasting, alumina" is a service in m2.
    "abrasive":           ("aluminium oxide production", "RoW", "aluminium oxide, non-metallurgical"),  # alumina grit (corundum)
    "grinding_wheel":     ("market for silicon carbide", "GLO", None),        # abrasive grain of wheels
    "polishing_compound": ("aluminium oxide production", "RoW", "aluminium oxide, non-metallurgical"),  # alumina powder
    "coolant":            ("market for lubricating oil", "RER", None),        # no emulsion dataset in ecoinvent
    "tool_material":      ("market for tungsten carbide powder", "GLO", None),  # cutting tool material
}

# Background activities of BAFU-2026 (ecoinvent v2 names). None = no dataset: the parameter is not modelled.
BAFU_FLOWS = {
    "electricity":        ("Electricity, medium voltage, at grid", "RER", None),       # consumption mix 2023
    "water":              ("Tap water, at user", "RER", None),
    "sodium_hydroxide":   ("Sodium hydroxide, 50% in H2O, production mix, at plant", "RER", None),  # per kg NaOH
    "hydrochloric_acid":  ("Hydrochloric acid, 30% in H2O, at plant", "RER", None),    # per kg HCl
    "nitric_acid":        ("Nitric acid, 50% in H2O, at plant", "RER", None),          # per kg HNO3
    "abrasive":           ("Aluminium oxide, at plant", "RER", None),                  # alumina proxy
    "grinding_wheel":     ("Silicon carbide, at plant", "RER", None),
    "polishing_compound": ("Aluminium oxide, at plant", "RER", None),
    "coolant":            ("Lubricating oil, at plant", "RER", None),
    "tool_material":      None,   # BAFU has no tungsten carbide: tool wear is not modelled
}
FLOWS = {"ecoinvent": ECOINVENT_FLOWS, "bafu": BAFU_FLOWS}[BACKGROUND]

# Prices per ecoinvent unit of each flow. INDICATIVE PLACEHOLDER PRICES except electricity:
# replace with supplier quotes. Chemicals are per kg of ACTIVE substance, as modelled
# (ecoinvent "without water" datasets), so they are higher than the diluted commercial price.
COST_FACTORS = {
    "electricity":        EUROSTAT_ELEC_PRICE_2025S1["RER"],   # EUR/kWh, Eurostat 2025-S1, band ID, EU27 average (0.164)
    "water":              0.004,  # EUR/kg, tap water with discharge (about 4 EUR/m3)
    "sodium_hydroxide":   0.50,   # EUR/kg
    "hydrochloric_acid":  0.25,   # EUR/kg
    "nitric_acid":        0.45,   # EUR/kg
    "abrasive":           1.50,   # EUR/kg, alumina grit
    "polishing_compound": 1.50,   # EUR/kg, alumina proxy
    "grinding_wheel":     2.50,   # EUR/kg, silicon carbide proxy
    "coolant":            2.00,   # EUR/kg, lubricating oil proxy
    "tool_material":      60.0,   # EUR/kg, tungsten carbide
}

# Regional analysis: EU27 member states and the United Kingdom (GB).
# Industrial electricity price of each country (EUR/kWh). GB: DESNZ (see l4m_lca/constants.py for the sources).
# Prices: REGIONAL_ELEC_PRICE, labour costs: regional_labour_costs() (file LABOUR_COST_FILE);
# both in l4m_lca. The EU27 average (REGIONAL_BASELINE = "RER") is the baseline.
REGIONAL_COUNTRIES = EU27_GB

# Labour cost per hour of each country (EUR/h), as in joining_task.ipynb: Eurostat lc_lci_lev 2025, NACE C,
# sheet "labour_cost" of country_data.xlsx; GB from l4m_lca/constants.py. The EU27 average is the baseline.
LABOUR_COSTS = regional_labour_costs()
LABOUR_COST = LABOUR_COSTS[REGIONAL_BASELINE]   # 35.0 EUR/h, EU27 average

# Labour hours per execution = process time (time_min or machining_time_min) x operator share.
# The operator share is the part of the process time that 1 operator works on the part.
CHEM_OPERATOR_SHARE = 0.25   # chemical baths: the operator loads and unloads, and does other work during immersion
MECH_OPERATOR_SHARE = 1.0    # mechanical treatments: the operator works on the part for the full time
OPERATOR_SHARE = {}          # override per process, for example {"pickling": 0.5}


In [3]:
# Chemical bath processes
INDUSTRIAL_TANK_SIZES_L = {
    "pickling": 10000,             # 10 m3, standard for joining preparation (ASTM A380)
    "alkaline_degreasing": 7500,   # 7.5 m3, common for degreasing
    "passivation": 7500,           # 7.5 m3, as degreasing
    "chemical_etching": 7500,      # 7.5 m3, as pickling
    "conversion_coating": 3000,    # 3 m3, smaller for dip coating
}
DEFAULT_TANK_SIZE_L = 7500

# Active chemical of each bath
ACTIVE_CHEMICAL = {
    "pickling": "hydrochloric_acid",
    "alkaline_degreasing": "sodium_hydroxide",
    "passivation": "nitric_acid",
    "chemical_etching": "sodium_hydroxide",
}

BATH_DENSITY_KG_PER_L = 1.0      # dilute process baths (water-like)
REPLENISHMENT_PER_H = 0.001      # fraction of the active chemical replenished per hour (losses, degradation)
ROOM_TEMP_C = 20
HEATER_EFFICIENCY = 0.95

# U-values (W/m2.K) of tank walls
U_VALUES = {
    "bare_steel": 9.8,          # uninsulated steel tank
    "stainless_steel": 8.5,
    "fiberglass_5cm": 0.57,     # 5 cm fiberglass insulation
    "polyurethane_5cm": 0.45,
    "typical_insulated": 0.6,
}
TANK_U_VALUE = U_VALUES["typical_insulated"]

In [4]:
# Pre-/post-joining process parameters with min/max ranges.
# Material-specific fields are keyed by "steel" / "aluminium".
# acid_concentration_% is the mass fraction of active chemical: 0.05 = 5 wt%.
CHEM_PREPOST_PARAMS = {
    "alkaline_degreasing": {
        "standard": {"steel": "ASTM B183", "aluminium": "ASTM B254"},
        "parameters": {
            "steel": {
                "acid_concentration_%": {"min": 0.05, "max": 0.1},
                "water_consumption_l": {"min": 5.0, "max": 15.0},
                "temperature_C": {"min": 60, "max": 80},
                "time_min": {"min": 5, "max": 15},
            },
            "aluminium": {
                "acid_concentration_%": {"min": 0.03, "max": 0.1},
                "water_consumption_l": {"min": 3.0, "max": 10.0},
                "temperature_C": {"min": 50, "max": 70},
                "time_min": {"min": 8, "max": 20},
            },
        },
        "description": {
            "steel": "Remove oils, grease, and contaminants from steel surface",
            "aluminium": "Remove oxides and contaminants using alkaline solutions with inhibitors",
        },
    },
    "chemical_etching": {
        "standard": "ASTM B253",
        "parameters": {
            "aluminium": {
                "acid_concentration_%": {"min": 0.05, "max": 0.10},
                "water_consumption_l": {"min": 5.0, "max": 15.0},
                "time_min": {"min": 2, "max": 10},
                "temperature_C": {"min": 50, "max": 65},
            },
        },
        "description": "Alkaline (NaOH) etch of aluminium to remove the natural oxide layer and a thin surface layer",
    },
    "pickling": {
        "standard": "ASTM A380, ISO 16048",
        "parameters": {
            "steel": {
                "acid_concentration_%": {"min": .1, "max": 0.2},
                "hydrofluoric_acid_consumption_%": {"min": 0.01, "max": 0.02},  # not modelled yet
                "water_consumption_l": {"min": 10.0, "max": 25.0},
                "time_min": {"min": 10, "max": 30},
                "temperature_C": {"min": 20, "max": 50},
            },
        },
        "description": "Remove weld scale and oxides using acid solutions",
    },
    "passivation": {
        "standard": "ASTM A380, ASTM A967",
        "parameters": {
            "steel": {
                "acid_concentration_%": {"min": 0.2, "max": 0.4},
                "water_consumption_l": {"min": 5.0, "max": 10.0},
                "time_min": {"min": 20, "max": 30},
                "temperature_C": {"min": 20, "max": 50},
            },
        },
        "description": "Form protective oxide layer on stainless steel surface",
    },
}

In [5]:
# Mechanical treatments. Each value is the total consumed over one execution.
MECH_PREPOST_PARAMS = {
    "mechanical_treatment": {
        "standard": {"steel": "ISO 8501-1 (surface preparation)", "aluminium": "ISO 8501-1"},
        "parameters": {
            "steel": {
                "abrasive_consumption_kg": {"min": 0.01, "max": 0.05},
                "electricity_kwh": {"min": 0.1, "max": 0.4},
                "time_min": {"min": 5, "max": 20},
                "surface_area_m2": {"min": 0.5, "max": 2.0},
            },
            "aluminium": {
                "abrasive_consumption_kg": {"min": 0.005, "max": 0.02},
                "coolant_consumption_l": {"min": 0.5, "max": 2.0},
                "electricity_kwh": {"min": 0.05, "max": 0.2},
                "time_min": {"min": 3, "max": 15},
                "surface_area_m2": {"min": 0.3, "max": 1.5},
            },
        },
        "condition": {"steel": "if oxides/scale are present"},
        "note": {"aluminium": "Use non-ferrous tools with coolant"},
        "description": {
            "steel": "Remove oxides and scale using abrasive methods",
            "aluminium": "Surface preparation using non-ferrous tools with mandatory coolant",
        },
    },
    "machining": {
        "standard": "ISO 3685 (tool life)",
        "parameters": {
            "steel": {
                "machining_time_min": {"min": 1, "max": 5},
                "tool_wear_pct": {"min": 5, "max": 15},
                "electricity_kwh": {"min": 0.05, "max": 0.2},
            },
        },
        "condition": "if tight tolerances are required",
        "description": "Precision machining for tight tolerance requirements",
    },
    "grinding_polishing": {
        "standard": {"steel": "ISO 16048, ASTM B912 (electropolishing)", "aluminium": "ISO 16048, ASTM B209"},
        "parameters": {
            "steel": {
                "grinding_wheel_consumption_kg": {"min": 0.005, "max": 0.02},
                "polishing_compound_kg": {"min": 0.01, "max": 0.05},
                "electricity_kwh": {"min": 0.1, "max": 0.5},
                "time_min": {"min": 5, "max": 25},
                "surface_roughness_um": {"min": 0.1, "max": 1.6},
            },
            "aluminium": {
                "grinding_wheel_consumption_kg": {"min": 0.002, "max": 0.01},
                "polishing_compound_kg": {"min": 0.005, "max": 0.02},
                "coolant_consumption_l": {"min": 0.1, "max": 0.5},
                "electricity_kwh": {"min": 0.05, "max": 0.2},
                "time_min": {"min": 3, "max": 15},
                "surface_roughness_um": {"min": 0.05, "max": 1.0},
            },
        },
        "condition": {"steel": "as needed for surface finish"},
        "note": {"aluminium": "Use non-ferrous tools with mandatory coolant"},
        "description": {
            "steel": "Achieve required surface finish and dimensional accuracy",
            "aluminium": "Final surface finishing using non-ferrous tools with coolant",
        },
    },
}

# Conversion of the mechanical parameters to ecoinvent units
COOLANT_DENSITY_KG_PER_L = 0.9     # density of the oil (lubricating oil proxy)
COOLANT_OIL_FRACTION = 0.05        # water-miscible coolant: emulsion with 5 % oil (5-9 % in practice)
COOLANT_LOSS_FRACTION = 0.10       # share of the coolant per execution that is lost (drag-out, mist) and topped up
TOOL_MASS_KG = 0.01   # about 10 g carbide indexable insert (typical 5-15 g), consumed at 100 % wear

# Parameter -> (flow, factor to the ecoinvent unit). Parameters without a flow are not modelled.
MECH_PARAMETER_FLOWS = {
    "electricity_kwh": ("electricity", 1.0),
    "abrasive_consumption_kg": ("abrasive", 1.0),
    "grinding_wheel_consumption_kg": ("grinding_wheel", 1.0),
    "polishing_compound_kg": ("polishing_compound", 1.0),
    "coolant_consumption_l": ("coolant", COOLANT_LOSS_FRACTION * COOLANT_OIL_FRACTION * COOLANT_DENSITY_KG_PER_L),  # kg oil per l
    "tool_wear_pct": ("tool_material", TOOL_MASS_KG / 100),   # % -> kg
}

## Project and background data

In [6]:
background = open_project(BG.project, BG.database)
flows = {name: find_activity(BG.database, *spec) if spec else None for name, spec in FLOWS.items()}
print(f"Background: {BG.database} (project {BG.project}), results in {BG.results}/")
display({name: f"{a['name']} [{a['location']}] ({a['unit']})" if a else "not modelled" for name, a in flows.items()})

methods = ef31_methods(INDICATORS)
units = method_units(methods)
impacts = BackgroundImpacts(methods, flows["electricity"])
prices = {flows[name].key: price for name, price in COST_FACTORS.items() if flows[name]}

Background: ecoinvent-3.12-cutoff (project Lasers4MaaS), results in results/


{'electricity': 'market group for electricity, medium voltage [RER] (kilowatt hour)',
 'water': 'market group for tap water [RER] (kilogram)',
 'sodium_hydroxide': 'market for sodium hydroxide, without water, in 50% solution state [RER] (kilogram)',
 'hydrochloric_acid': 'market for hydrochloric acid, without water, in 30% solution state [RER] (kilogram)',
 'nitric_acid': 'market for nitric acid, without water, in 50% solution state [RER w/o RU] (kilogram)',
 'abrasive': 'aluminium oxide production [RoW] (kilogram)',
 'grinding_wheel': 'market for silicon carbide [GLO] (kilogram)',
 'polishing_compound': 'aluminium oxide production [RoW] (kilogram)',
 'coolant': 'market for lubricating oil [RER] (kilogram)',
 'tool_material': 'market for tungsten carbide powder [GLO] (kilogram)'}

/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 3.43e+12)
  warnings.warn(msg, UmfpackWarning)


## Chemical pre- and post-joining processes

- Active chemical (kg of active substance): concentration x bath mass x replenished fraction per hour x hours.
- Electricity (kWh): heat that replaces the tank heat loss during the process time.
- Water (kg): rinsing water.
- Labour (h): process time x `CHEM_OPERATOR_SHARE`.

In [7]:
chem_data = {}
for service, material, params, _ in per_material(CHEM_PREPOST_PARAMS):
    code = process_code(service, material)
    chem_data[(CHEM_PREPOST_DATABASE, code)] = {
        "name": process_name(service, material), "unit": "unit", "location": "NL",
        "exchanges": chemical_exchanges(
            CHEM_PREPOST_DATABASE, code, params, flows[ACTIVE_CHEMICAL[service]], flows["electricity"], flows["water"],
            tank_volume_l=INDUSTRIAL_TANK_SIZES_L.get(service, DEFAULT_TANK_SIZE_L),
            replenishment_per_h=REPLENISHMENT_PER_H, bath_density_kg_per_l=BATH_DENSITY_KG_PER_L,
            room_temp_c=ROOM_TEMP_C, u_value=TANK_U_VALUE, heater_efficiency=HEATER_EFFICIENCY),
        LABOUR_FIELD: prepost_labour(params, OPERATOR_SHARE.get(service, CHEM_OPERATOR_SHARE)),
    }
chem = bd.Database(CHEM_PREPOST_DATABASE)
chem.write(chem_data)

  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:00<00:00, 7701.62it/s]

21:49:57+0100 [info     ] Vacuuming database            


## Mechanical pre- and post-joining processes

Labour (h): process time x `MECH_OPERATOR_SHARE`.

In [8]:
parameter_flows = {p: (flows[f], factor) for p, (f, factor) in MECH_PARAMETER_FLOWS.items()}
mech_data = {}
for service, material, params, _ in per_material(MECH_PREPOST_PARAMS):
    code = process_code(service, material)
    exchanges, ignored = mechanical_exchanges(MECH_PREPOST_DATABASE, code, params, parameter_flows)
    ignored = [p for p in ignored if p not in TIME_PARAMETERS]   # the process time gives the labour hours
    print(f"{code}: not modelled (no ecoinvent flow): {ignored}")
    mech_data[(MECH_PREPOST_DATABASE, code)] = {
        "name": process_name(service, material), "unit": "unit", "location": "NL", "exchanges": exchanges,
        LABOUR_FIELD: prepost_labour(params, OPERATOR_SHARE.get(service, MECH_OPERATOR_SHARE))}
mech = bd.Database(MECH_PREPOST_DATABASE)
mech.write(mech_data)

mechanical_treatment_steel: not modelled (no ecoinvent flow): ['surface_area_m2']
mechanical_treatment_aluminium: not modelled (no ecoinvent flow): ['surface_area_m2']
machining_steel: not modelled (no ecoinvent flow): []
grinding_polishing_steel: not modelled (no ecoinvent flow): ['surface_roughness_um']
grinding_polishing_aluminium: not modelled (no ecoinvent flow): ['surface_roughness_um']


  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:00<00:00, 29046.43it/s]

21:50:06+0100 [info     ] Vacuuming database            


## LCIA results

Impacts and operating costs of each process per execution, in the minimum, amount and maximum scenarios.
The operating costs are the consumable costs, the personnel costs (EU27 average labour cost) and the nominal OPEX (their sum).

In [9]:
lcia_df = scenario_table([chem, mech], impacts, prices, wage=LABOUR_COST)
export(lcia_df, units, BG.results, PREPOST_STEM)
lcia_df

Results saved to results/Chemical_Pre_and_Post_processes_Mechanical_Pre_and_Post_processes_LCIA_results.csv and results/Chemical_Pre_and_Post_processes_Mechanical_Pre_and_Post_processes_LCIA_results.json


,Database,Process,Key,Scenario,acidification,eutrophication: terrestrial,human toxicity: carcinogenic,"human toxicity: carcinogenic, inorganics","human toxicity: carcinogenic, organics",particulate matter formation,photochemical oxidant formation: human health,climate change: total (EF v3.1 - IPCC 2021),Electricity Consumption (kWh),Consumable costs (EUR),Personnel costs (EUR),Nominal OPEX (EUR)
0,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for aluminium,alkaline_degreasing_aluminium,minimum,0.000236,0.000418,2.299207e-10,7.183149e-12,2.227376e-10,1.443271e-09,0.000129,0.052393,0.058078,0.036525,1.166667,1.203191
1,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for aluminium,alkaline_degreasing_aluminium,amount,0.000884,0.001618,1.063749e-09,2.794041e-11,1.035809e-09,5.891000e-09,0.000495,0.198565,0.150035,0.120606,2.041667,2.162272
2,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for aluminium,alkaline_degreasing_aluminium,maximum,0.001532,0.002818,1.897577e-09,4.869767e-11,1.848880e-09,1.033873e-08,0.000862,0.344737,0.241992,0.204687,2.916667,3.121353
3,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for steel,alkaline_degreasing_steel,minimum,0.000228,0.000410,2.393686e-10,7.337413e-12,2.320312e-10,1.467840e-09,0.000126,0.050807,0.048398,0.043562,0.729167,0.772729
4,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for steel,alkaline_degreasing_steel,amount,0.000725,0.001320,8.334472e-10,2.318054e-11,8.102666e-10,4.786442e-09,0.000404,0.162395,0.133096,0.116515,1.458333,1.574849
5,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for steel,alkaline_degreasing_steel,maximum,0.001223,0.002230,1.427526e-09,3.902366e-11,1.388502e-09,8.105044e-09,0.000683,0.273983,0.217793,0.189468,2.187500,2.376968
6,Chemical_Pre_and_Post_processes.9a629b55,chemical_etching for aluminium,chemical_etching_aluminium,minimum,0.000088,0.000161,9.637111e-11,3.231005e-12,9.314011e-11,6.147873e-10,0.000049,0.019523,0.014520,0.028631,0.291667,0.320298
7,Chemical_Pre_and_Post_processes.9a629b55,chemical_etching for aluminium,chemical_etching_aluminium,amount,0.000425,0.000784,5.237353e-10,1.435667e-11,5.093786e-10,2.949906e-09,0.000240,0.095338,0.061708,0.084495,0.875000,0.959495
8,Chemical_Pre_and_Post_processes.9a629b55,chemical_etching for aluminium,chemical_etching_aluminium,maximum,0.000762,0.001408,9.510995e-10,2.548233e-11,9.256172e-10,5.285025e-09,0.000431,0.171154,0.108897,0.140359,1.458333,1.598692
9,Chemical_Pre_and_Post_processes.9a629b55,passivation for steel,passivation_steel,minimum,0.004543,0.018543,8.128978e-11,4.033078e-11,4.095900e-11,2.285948e-08,0.001101,0.505927,0.000000,0.245000,2.916667,3.161667


## Regional analysis: EU and UK electricity markets

Replaces the RER electricity market with the market of each EU member state and of the United Kingdom (GB). Each country also gets its own labour cost (personnel costs).
The map shows the impacts and operating costs per execution of each process.
Water, chemicals and abrasives keep their ecoinvent datasets and prices.
Countries without a Eurostat labour cost (NL, BE) have empty personnel costs and nominal OPEX (NaN).
The file `results/regional_prepost.csv` is an input of `workflow_assembly.ipynb`.

In [10]:
markets = electricity_markets(background, REGIONAL_COUNTRIES, BG.country_electricity)
regional_df = regional_table([chem, mech], impacts, markets, REGIONAL_ELEC_PRICE, prices, REGIONAL_BASELINE,
                           wages=LABOUR_COSTS)
regional_df.to_csv(f"{BG.results}/regional_prepost.csv", index=False)
regional_df.head()

/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 3.43e+12)
  warnings.warn(msg, UmfpackWarning)


,database,process,key,scenario,country,indicator,impact
0,Chemical_Pre_and_Post_processes,alkaline_degreasing for aluminium,alkaline_degreasing_aluminium,minimum,RER,acidification,2.357450e-04
1,Chemical_Pre_and_Post_processes,alkaline_degreasing for aluminium,alkaline_degreasing_aluminium,minimum,RER,eutrophication: terrestrial,4.184470e-04
2,Chemical_Pre_and_Post_processes,alkaline_degreasing for aluminium,alkaline_degreasing_aluminium,minimum,RER,human toxicity: carcinogenic,2.299207e-10
3,Chemical_Pre_and_Post_processes,alkaline_degreasing for aluminium,alkaline_degreasing_aluminium,minimum,RER,"human toxicity: carcinogenic, inorganics",7.183149e-12
4,Chemical_Pre_and_Post_processes,alkaline_degreasing for aluminium,alkaline_degreasing_aluminium,minimum,RER,"human toxicity: carcinogenic, organics",2.227376e-10


In [11]:
display(plots.regional_explorer(
    regional_df,
    filters=[("Process", "process", sorted(regional_df["process"].unique()), None),
             ("Scenario", "scenario", list(SCENARIOS), "amount")],
    units={**units, **COST_UNITS},
    title=lambda f: f"{f['process']} ({f['scenario']}): per execution",
    baseline=REGIONAL_BASELINE, countries=REGIONAL_COUNTRIES,
))